<a href="https://colab.research.google.com/github/Srinidhi08092006/codesoft/blob/main/aristoverse_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate torch sentencepiece

In [2]:
!pip install transformers accelerate torch sentencepiece

In [3]:
!pip install -U bitsandbytes accelerate transformers


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [1]:
from google.colab import files
import numpy as np
import os
import pandas as pd
import gc
import torch

# Handle file upload
uploaded = files.upload()
if not uploaded:
    print("No file uploaded.")
else:
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    output_dir = "/content/output"
    os.makedirs(output_dir, exist_ok=True)

    def standardize(row):
        raw = row["RawValue"]
        mean, sd = row.get("Mean", np.nan), row.get("SD", np.nan)
        low, high = row.get("RefLow", np.nan), row.get("RefHigh", np.nan)
        flags, z, method = [], np.nan, "UNSTANDARDIZED"
        if pd.isna(low) or pd.isna(high) or high <= low: flags.append("MISSING_RANGE")
        if not pd.isna(sd) and sd <= 0: flags.append("INVALID_SD")
        if not pd.isna(mean) and not pd.isna(sd) and sd > 0:
            z = (raw - mean) / sd
            method = "Z_SCORE"
        elif not pd.isna(low) and not pd.isna(high) and high > low:
            mean_rr, sd_rr = (low + high) / 2, (high - low) / 4
            if sd_rr > 0:
                z = (raw - mean_rr) / sd_rr
                method = "REF_RANGE_Z"
            else:
                z = (raw - low) / (high - low)
                method = "MIN_MAX"
        if not pd.isna(z) and abs(z) >= 3: flags.append("EXTREME_Z")
        return pd.Series([z, method, ";".join(flags)])

    df[["ZScore", "MethodUsed", "Flags"]] = df.apply(standardize, axis=1)
    df.to_csv(os.path.join(output_dir, "M05_lab_zscores.csv"), index=False)
    print("✅ Standardization completed")

    # Clear memory
    gc.collect()
    torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    # Consider 'Qwen/Qwen2.5-3B-Instruct' if 7B is too slow with offloading
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True # Enable offloading for limited RAM
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )

        allowed_actions = ["search_web", "retrieve_document", "summarize_text", "translate_text", "classify_email", "generate_report", "extract_entities", "answer_question", "store_record", "delete_record"]
        prompt = f"USER REQUEST: \"Summarize the uploaded research paper.\"\nALLOWED ACTIONS: {allowed_actions}\nOutput ONLY valid JSON."

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(response)

    except Exception as e:
        print(f"❌ Error: {e}")

Saving M05_Processed_1500rows.csv to M05_Processed_1500rows (2).csv
✅ Standardization completed


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


USER REQUEST: "Summarize the uploaded research paper."
ALLOWED ACTIONS: ['search_web', 'retrieve_document', 'summarize_text', 'translate_text', 'classify_email', 'generate_report', 'extract_entities', 'answer_question', 'store_record', 'delete_record']
Output ONLY valid JSON. DO NOT include any explanations. 
```json
{
  "action": "summarize_text"
}
``` ```json
{
  "action": "summarize_text"
}
```


In [2]:
import pandas as pd
import numpy as np

# Load your original CSV
df = pd.read_csv("/content/output/M05_lab_zscores.csv")

# Add missing columns if they don't exist
if "ZScore" not in df.columns:
    df["ZScore"] = np.nan  # empty ZScore for now
if "Flags" not in df.columns:
    df["Flags"] = ""  # empty Flags

# Optional: mark extreme values
df.loc[df["RawValue"] > 200, "Flags"] = "EXTREME_Z"

# Save the fixed CSV
df.to_csv("/content/output/M05_lab_zscores_fixed.csv", index=False)
print("✅ CSV fixed and ready for Task-3")
csv_path = "/content/output/M05_lab_zscores.csv"
csv_path = "/content/output/M05_lab_zscores_fixed.csv"
# =============================
# TASK-3: Neurocritical Recommendation & Guidance
# =============================
import os
import json
import pandas as pd
import numpy as np

# -----------------------------
# Ensure output directory
# -----------------------------
os.makedirs("/content/output", exist_ok=True)

# -----------------------------
# Load standardized lab data
# -----------------------------
csv_path = "/content/output/M05_lab_zscores.csv"

try:
    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError(f"{csv_path} is empty!")
except Exception as e:
    print(f"Error loading CSV: {e}")
    df = pd.DataFrame(columns=["CanonicalName", "RawValue", "ZScore", "Flags"])

# -----------------------------
# Neuro Risk Classification
# -----------------------------
def classify_neuro_risk(z, flags):
    if "EXTREME_Z" in str(flags):
        return "CRITICAL"
    if pd.isna(z):
        return "UNKNOWN"
    z = abs(z)
    if z < 1.5:
        return "NORMAL"
    elif z < 2.5:
        return "MILD"
    elif z < 3.5:
        return "HIGH"
    else:
        return "CRITICAL"

df["NeuroRisk"] = df.apply(
    lambda r: classify_neuro_risk(r["ZScore"], r["Flags"]),
    axis=1
)

# -----------------------------
# Clinical Fallback Rules
# -----------------------------
def clinical_fallback(biomarker, value):
    if pd.isna(value):
        return "UNKNOWN"
    if biomarker == "Cholesterol":
        if value >= 240:
            return "HIGH"
        elif value >= 200:
            return "MODERATE"
        else:
            return "LOW"
    if biomarker == "Hemoglobin":
        if value < 10:
            return "HIGH"
        elif value < 12:
            return "MODERATE"
        else:
            return "LOW"
    if biomarker == "Glucose":
        if value >= 126:
            return "HIGH"
        elif value >= 100:
            return "MODERATE"
        else:
            return "LOW"
    if biomarker == "VitaminD":
        if value < 20:
            return "HIGH"
        elif value < 30:
            return "MODERATE"
        else:
            return "LOW"
    if biomarker == "HeartRate":
        if value > 120:
            return "HIGH"
        elif value > 100:
            return "MODERATE"
        else:
            return "LOW"
    return "UNKNOWN"

# Apply fallback if ZScore is NaN
df["FinalRisk"] = df.apply(
    lambda r: clinical_fallback(r["CanonicalName"], r["RawValue"])
    if pd.isna(r["ZScore"]) else r["NeuroRisk"],
    axis=1
)

# -----------------------------
# Guidance (NOT dosage)
# -----------------------------
def guidance(biomarker, risk):
    if biomarker == "Hemoglobin" and risk in ["HIGH", "MODERATE"]:
        return "Review anemia management and nutritional support (clinician-guided)"
    if biomarker == "Glucose" and risk in ["HIGH", "MODERATE"]:
        return "Glycemic monitoring and lifestyle optimization advised"
    if biomarker == "VitaminD" and risk in ["HIGH", "MODERATE"]:
        return "Assess vitamin D sufficiency and supplementation needs"
    if biomarker == "Cholesterol" and risk in ["HIGH", "MODERATE"]:
        return "Lipid profile monitoring and dietary counseling recommended"
    if biomarker == "HeartRate" and risk in ["HIGH", "MODERATE"]:
        return "Monitor autonomic stability and cardiovascular stress"
    return "No immediate neurocritical action required"

df["Guidance"] = df.apply(
    lambda r: guidance(r["CanonicalName"], r["FinalRisk"]),
    axis=1
)

# -----------------------------
# Aggregate by biomarker
# -----------------------------
agg = df.groupby("CanonicalName").agg(
    MeanValue=("RawValue", "mean"),
    MaxRisk=("FinalRisk", lambda x: x.value_counts().idxmax() if not x.empty else "UNKNOWN")
).reset_index()

# -----------------------------
# Overall Neurocritical Risk
# -----------------------------
def overall_risk(risks):
    if "CRITICAL" in risks.values:
        return "CRITICAL"
    if list(risks.values).count("HIGH") >= 2:
        return "HIGH"
    if "MODERATE" in risks.values:
        return "MODERATE"
    return "LOW"

overall_neuro_risk = overall_risk(agg["MaxRisk"]) if not agg.empty else "UNKNOWN"
print("✅ Overall Neurocritical Risk:", overall_neuro_risk)

# -----------------------------
# Save CSV Output
# -----------------------------
df.to_csv("/content/output/M05_neurocritical_guidance.csv", index=False)

# -----------------------------
# Save JSON Summary (Safe)
# -----------------------------
summary = {
    "overall_neurocritical_risk": str(overall_neuro_risk),
    "biomarkers_analyzed": agg.to_dict(orient="records"),
    "safety_note": "Decision-support only. Not a medical prescription."
}

with open("/content/output/M05_neurocritical_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)  # default=str ensures NumPy types don't break JSON

# -----------------------------
# Save Markdown Report
# -----------------------------
with open("/content/output/M05_neurocritical_recommendations.md", "w") as f:
    f.write("# Neurocritical Recommendation\n\n")
    f.write(f"**Overall Risk:** {overall_neuro_risk}\n\n")
    for _, r in agg.iterrows():
        f.write(
            f"- **{r['CanonicalName']}** | Mean={r['MeanValue']:.2f} | Risk={r['MaxRisk']}\n"
        )

# -----------------------------
# Prompt & Model Info
# -----------------------------
prompt = ("Analyze this lab dataset for neurocritical risk and generate structured outputs "
          "(CSV, JSON, Markdown). Ensure handling of long-context data, schema correctness, "
          "and consistent risk aggregation.")

with open("/content/output/prompt.txt", "w") as f:
    f.write(prompt)

model_info = {
    "model_name": "Command-R (or equivalent open long-context model)",
    "model_source": "HuggingFace",
    "model_type": "Long-context instruction-following LLM",
    "evaluation_focus": [
        "long-context handling",
        "action whitelist compliance",
        "JSON/CSV/Markdown structured output correctness"
    ]
}

with open("/content/output/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("✅ All Task-3 neurocritical outputs generated successfully")
import pandas as pd
import numpy as np

# Load your CSV
df = pd.read_csv("/content/output/M05_lab_zscores.csv")

# Add missing columns
if "ZScore" not in df.columns:
    df["ZScore"] = np.nan
if "Flags" not in df.columns:
    df["Flags"] = ""

# Optional: mark any extreme values
df.loc[df["RawValue"] > 200, "Flags"] = "EXTREME_Z"  # example

# Save corrected CSV
df.to_csv("/content/output/M05_lab_zscores_fixed.csv", index=False)
import pandas as pd

df = pd.read_csv("/content/output/M05_lab_zscores_fixed.csv")
print(df.head())
print(df.dtypes)
print(df.isna().sum())
df["RawValue"] = pd.to_numeric(df["RawValue"], errors="coerce")
df["ZScore"].fillna(0, inplace=True)  # or your computed ZScore
agg = df.groupby("CanonicalName").agg(
    MeanValue=("RawValue", "mean"),
    MaxRisk=("FinalRisk", lambda x: x.value_counts().idxmax() if not x.empty else "UNKNOWN")
).reset_index()

print(agg)


✅ CSV fixed and ready for Task-3
✅ Overall Neurocritical Risk: LOW
✅ All Task-3 neurocritical outputs generated successfully
  CanonicalName  RawValue   Unit  LowerLimit  UpperLimit     SD  \
0      VitaminD     15.74  ng/mL         NaN         NaN    NaN   
1     HeartRate     54.76    bpm        60.0       100.0  10.00   
2   Cholesterol    228.05  mg/dL       125.0       200.0  18.75   
3   Cholesterol    137.54  mg/dL       125.0       200.0  18.75   
4     HeartRate    106.69    bpm        60.0       100.0  10.00   

   RawValue_ZScore  LowerLimit_ZScore  UpperLimit_ZScore  SD_ZScore  \
0        -1.068499                NaN                NaN        NaN   
1        -0.431865          -0.172351          -0.062669   0.109242   
2         2.395464           1.448853           1.465898   1.469731   
3         0.918740           1.448853           1.465898   1.469731   
4         0.415404          -0.172351          -0.062669   0.109242   

  RawValue_Flag LowerLimit_Flag UpperLimit_Fl

/tmp/ipython-input-2669756517.py:241: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["ZScore"].fillna(0, inplace=True)  # or your computed ZScore


In [3]:
print(df.columns.tolist())


['CanonicalName', 'RawValue', 'Unit', 'LowerLimit', 'UpperLimit', 'SD', 'RawValue_ZScore', 'LowerLimit_ZScore', 'UpperLimit_ZScore', 'SD_ZScore', 'RawValue_Flag', 'LowerLimit_Flag', 'UpperLimit_Flag', 'SD_Flag', 'NeuroRisk', 'FinalRisk', 'Guidance', 'ZScore', 'MethodUsed', 'Flags']
